# Train Kikuyu VITS From Scratch (WaxalNLP `kik_tts`)

This notebook runs with a dedicated Python 3.11 virtual environment because `trainer` and `TTS==0.22.0` are not compatible with newer Python versions.


In [ ]:
!python --version
!sudo apt-get update
!sudo apt-get install -y python3.11 python3.11-venv
!python3.11 -m venv /content/tts311
!bash -lc "source /content/tts311/bin/activate && python -m pip install -U pip"
!bash -lc "source /content/tts311/bin/activate && python -m pip install datasets[audio] soundfile librosa pyyaml huggingface_hub"
!bash -lc "source /content/tts311/bin/activate && python -m pip install coqpit trainer TTS==0.22.0"
!bash -lc "source /content/tts311/bin/activate && python --version"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!git clone https://github.com/kihahu/kikuyu-tts.git
%cd /content/kikuyu-tts


In [ ]:
!bash -lc "source /content/tts311/bin/activate && python scripts/prepare_waxal_kik_tts.py \
  --dataset-name google/WaxalNLP \
  --dataset-config kik_tts \
  --split train \
  --output-dir data/waxal_kik_tts \
  --target-sample-rate 16000 \
  --min-duration-sec 0.6 \
  --max-duration-sec 25.0 \
  --min-rms 0.0035 \
  --seed 42 \
  --dev-ratio 0.10 \
  --test-ratio 0.05"


In [ ]:
!bash -lc "source /content/tts311/bin/activate && python scripts/build_kikuyu_vocab.py \
  --train-manifest data/waxal_kik_tts/manifests/train.jsonl \
  --dev-manifest data/waxal_kik_tts/manifests/dev.jsonl \
  --out-dir artifacts/tokenizer_kikuyu_char"


In [ ]:
%cd /content
!git clone https://github.com/coqui-ai/TTS.git
%cd /content/kikuyu-tts


In [ ]:
# Start fresh
!bash -lc "source /content/tts311/bin/activate && python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/TTS"


In [ ]:
# Resume mode
!bash -lc "source /content/tts311/bin/activate && python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/TTS \
  --resume"


In [ ]:
# Optional: push checkpoints to HF Hub
!bash -lc "source /content/tts311/bin/activate && huggingface-cli login"
!bash -lc "source /content/tts311/bin/activate && python scripts/colab_train_vits_scratch.py \
  --config configs/train_kikuyu_vits_scratch_colab.yaml \
  --trainer-repo /content/TTS \
  --resume \
  --push-hf"


Create a metrics CSV at `artifacts/checkpoint_metrics.csv` with columns:
- `checkpoint`
- `synthesis_success_rate`
- `clipping_rate`
- `mos_lite`
- `wer_proxy`


In [ ]:
!bash -lc "source /content/tts311/bin/activate && python scripts/evaluate_and_select.py \
  --metrics-csv artifacts/checkpoint_metrics.csv \
  --out-json artifacts/best_checkpoint_selection.json \
  --out-csv artifacts/tts_eval_summary.csv"


In [ ]:
!bash -lc "source /content/tts311/bin/activate && python scripts/prepare_local_integration.py \
  --best-checkpoint-dir artifacts/colab_runs/kikuyu_vits_scratch/checkpoint_best \
  --tokenizer-dir artifacts/tokenizer_kikuyu_char \
  --eval-summary-csv artifacts/tts_eval_summary.csv \
  --out-dir artifacts/local_integration/kikuyu_vits_best \
  --model-id kikuyu-vits-scratch-waxal"
